## The client / daemon architecture

When you type `docker run`, the `docker` you invoked is just a small **client**. It does no container work itself — it packages your command as a REST API call and sends it, over a Unix socket (`/var/run/docker.sock`) or a TCP port, to a long-running **daemon**. Understanding this split explains almost every "why did that happen" in Docker.

```
+-----------------+   REST API    +----------------------+
| docker CLI      | ------------> | dockerd (daemon)     |
| (your terminal) | docker.sock   |  images · networks   |
+-----------------+               |  volumes · build     |
                                  +----------+-----------+
                                             | gRPC
                                             v
                                  +----------------------+
                                  | containerd (runtime  |
                                  |  supervisor)         |
                                  +----------+-----------+
                                             | spawns
                                             v
                                       +-----------+
                                       |   runc    |  ns + cgroups
                                       +-----------+
```

**The daemon (`dockerd`)** is the brain: it manages images, networks, volumes, and builds, and holds all state. Kill the daemon and every `docker` command fails, even though your container processes may keep running.

**`containerd`** is the daemon's runtime supervisor — an industry-standard component (a CNCF project) that actually tracks the lifecycle of running containers. `dockerd` delegates to it over gRPC.

**`runc`** is the low-level tool that does the final act: it talks to the kernel to create the namespaces and cgroups, then execs your process as the container's PID 1. It runs, hands off, and exits — the container keeps going.

Two consequences worth holding onto:

- **The client is thin and remote-able.** Point `DOCKER_HOST` at another machine's daemon and the same CLI drives it — the client never needed to be on the same box.
- **The daemon runs as root** (it must, to create namespaces). That's why socket access effectively equals root on the host — a security point module 08 returns to.